In [1]:
import numpy as np

In [2]:
class RNN:
    def __init__(self, input_size, hidden_size, output_size):
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.W_hy = np.random.randn(output_size, hidden_size) * 0.01
        
        self.b_h = np.zeros((hidden_size, 1))
        self.b_y = np.zeros((output_size, 1))

    def forward(self, inputs):
        self.inputs = inputs
        self.h_states = {0: np.zeros((self.W_hh.shape[0], 1))}
        
        for t, x in enumerate(inputs):
            self.h_states[t + 1] = np.tanh(np.dot(self.W_xh, x) + np.dot(self.W_hh, self.h_states[t]) + self.b_h)
        
        self.y = np.dot(self.W_hy, self.h_states[len(inputs)]) + self.b_y
        return self.y

    def backward(self, d_y, learning_rate):
        d_W_hy = np.dot(d_y, self.h_states[len(self.inputs)].T)
        d_b_y = d_y
        
        d_h = np.dot(self.W_hy.T, d_y)
        
        d_W_hh = np.zeros_like(self.W_hh)
        d_W_xh = np.zeros_like(self.W_xh)
        d_b_h = np.zeros_like(self.b_h)
        
        for t in reversed(range(len(self.inputs))):
            temp = (1 - self.h_states[t + 1] ** 2) * d_h  
            
            d_b_h += temp
            d_W_hh += np.dot(temp, self.h_states[t].T)
            d_W_xh += np.dot(temp, self.inputs[t].T)
            
            d_h = np.dot(self.W_hh.T, temp)
            
        for d in [d_W_xh, d_W_hh, d_W_hy, d_b_h, d_b_y]:
            np.clip(d, -1, 1, out=d)
            
        self.W_hy -= learning_rate * d_W_hy
        self.b_y -= learning_rate * d_b_y
        self.W_hh -= learning_rate * d_W_hh
        self.W_xh -= learning_rate * d_W_xh
        self.b_h -= learning_rate * d_b_h

In [3]:
X_train = [
    [np.array([[0.1]]), np.array([[0.2]]), np.array([[0.3]])],
    [np.array([[0.4]]), np.array([[0.5]]), np.array([[0.6]])],
    [np.array([[0.2]]), np.array([[0.2]]), np.array([[0.2]])]
]
Y_train = [np.array([[0.6]]), np.array([[1.5]]), np.array([[0.6]])]

In [4]:
rnn = RNN(input_size=1, hidden_size=16, output_size=1)
epochs = 500
learning_rate = 0.05

In [5]:
for epoch in range(epochs):
    total_loss = 0
    for x, y_true in zip(X_train, Y_train):
        y_pred = rnn.forward(x)
        
        loss = np.mean((y_pred - y_true) ** 2)
        total_loss += loss
        
        d_y = 2 * (y_pred - y_true)
        
        rnn.backward(d_y, learning_rate)
        
    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {total_loss/len(X_train):.6f}")

Epoch 0 | Loss: 0.903446
Epoch 100 | Loss: 0.090896
Epoch 200 | Loss: 0.002512
Epoch 300 | Loss: 0.001441
Epoch 400 | Loss: 0.000807


In [6]:
test_seq = [np.array([[0.3]]), np.array([[0.1]]), np.array([[0.4]])] 
pred = rnn.forward(test_seq)

In [7]:
print(f"Input Sequence: [0.3, 0.1, 0.4]")
print(f"Actual Expected Output: 0.8000")
print(f"RNN Predicted Output: {pred[0][0]:.4f}")

Input Sequence: [0.3, 0.1, 0.4]
Actual Expected Output: 0.8000
RNN Predicted Output: 0.8245
